# Install package

In [2]:
!pip install pygamma-agreement

In [3]:
!pip install "pygamma-agreement[CBC]"

# Google Drive

In [4]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Imports

In [5]:
import json
import os
import pandas as pd
import numpy as np
from pygamma_agreement import Continuum
from pyannote.core import Segment
from pygamma_agreement import CombinedCategoricalDissimilarity

# Find the drive folder with all the files

In [6]:
os.listdir("/content/drive/MyDrive/Zip files/ALL_JSON_files_MAthesis")

['exportedproject.json',
 'event.log',
 'annotation',
 'log',
 'source',
 'annotation_ser']

# JSON parser


In [7]:
def parse_inception_json(filepath, annotator_name):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    feature_structures = data["%FEATURE_STRUCTURES"]

    # Map internal INCEpTION names to readable label names
    # Example: IClaim -> C-IMP, Narrative -> Exordium
    feature_name_to_ui_name = {}

    for fs in feature_structures:
        if fs.get("%TYPE") == "de.tudarmstadt.ukp.clarin.webanno.api.type.FeatureDefinition":
            internal_name = fs.get("name")
            readable_name = fs.get("uiName")

            if internal_name and readable_name:
                feature_name_to_ui_name[internal_name] = readable_name

    # These are text helper fields, not span labels for gamma
    ignore_features = {
        "IClaimtext",
        "MCIMPtext",
        "PIMPtext"
    }

    # These are the actual annotation layers
    annotation_layers = {
        "webanno.custom.Argumentation",
        "webanno.custom.Narration"
    }

    annotations = []

    for fs in feature_structures:
        if fs.get("%TYPE") not in annotation_layers:
            continue

        start = fs.get("begin")
        end = fs.get("end")

        if start is None or end is None:
            continue

        for feature_name, value in fs.items():
            if feature_name.startswith("%") or feature_name.startswith("@"):
                continue

            if feature_name in {"begin", "end"}:
                continue

            if feature_name in ignore_features:
                continue

            if value is True:
                label = feature_name_to_ui_name.get(feature_name, feature_name)

                annotations.append({
                    "annotator": annotator_name,
                    "start": int(start),
                    "end": int(end),
                    "label": label
                })

    return annotations

In [8]:
annotation_root = "/content/drive/MyDrive/Zip files/ALL_JSON_files_MAthesis/annotation"

# Define label groups for argumentation layer and narration layer

In [9]:
argumentation_labels = {
    "C-EXP",
    "MC-EXP",
    "C-IMP",
    "MC-IMP",
    "P-EXP",
    "P-IMP"
}

narration_labels = {
    "Confirmatio",
    "Narratio",
    "Exordium",
    "Other",
    "Peroratio",
    "Propositio"
    "NONE"
}

# Helper function
 So we can use it for all gamma tests: overall, argumentation, narration, standard, soft, and stronger label penalty.

In [10]:
def compute_gamma_test(
    annotations,
    layer_name="overall",
    selected_labels=None,
    alpha=1,
    beta=1,
    soft=False,
    seed=42,
    add_none_for_missing_narration=False
):
    continuum = Continuum()

    selected_annotations = [
        ann for ann in annotations
        if selected_labels is None or ann["label"] in selected_labels
    ]

    # Special case: narration layer where one annotator has no narration annotations.
    # We add a dummy "none" span so gamma can still compare two annotators.
    if add_none_for_missing_narration:
        annotators_present = {ann["annotator"] for ann in selected_annotations}
        all_annotators = {ann["annotator"] for ann in annotations}

        missing_annotators = all_annotators - annotators_present

        for annotator in missing_annotators:
            selected_annotations.append({
                "annotator": annotator,
                "start": 0,
                "end": 1,
                "label": "none"
            })

    for ann in selected_annotations:
        continuum.add(
            ann["annotator"],
            Segment(ann["start"], ann["end"]),
            ann["label"]
        )

    dissimilarity = CombinedCategoricalDissimilarity(
        alpha=alpha,
        beta=beta
    )

    np.random.seed(seed)

    gamma_result = continuum.compute_gamma(
        dissimilarity,
        soft=soft
    )

    return {
        "layer": layer_name,
        "gamma_type": "soft" if soft else "standard",
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma_result.gamma
    }

# TED folders with both annotations

In [11]:
document_pairs = []

# Go through every folder inside annotation/
for folder_name in os.listdir(annotation_root):

    folder_path = os.path.join(annotation_root, folder_name)

    # Only keep folders
    if not os.path.isdir(folder_path):
        continue

    # Build paths to both annotators
    luiza_file = os.path.join(folder_path, "filip.json")
    sara_file = os.path.join(folder_path, "nabhani.json")

    # Only keep folders where BOTH files exist
    if os.path.exists(luiza_file) and os.path.exists(sara_file):

        document_pairs.append({
            "document": folder_name.replace(".txt", ""),
            "luiza_file": luiza_file,
            "sara_file": sara_file
        })

print("Documents found:", len(document_pairs))


Documents found: 20


# Compute Gamma for everything

In [12]:
all_results = []

for pair in document_pairs:
    document_id = pair["document"]

    print("Processing:", document_id)

    luiza_annotations = parse_inception_json(
        pair["luiza_file"],
        "Luiza"
    )

    sara_annotations = parse_inception_json(
        pair["sara_file"],
        "Sara"
    )

    all_annotations = luiza_annotations + sara_annotations

    gamma_tests = [

        # Overall standard gamma
        compute_gamma_test(
            all_annotations,
            layer_name="overall",
            selected_labels=None,
            alpha=1,
            beta=1,
            soft=False),

        # Overall soft gamma
        compute_gamma_test(
            all_annotations,
            layer_name="overall",
            selected_labels=None,
            alpha=1,
            beta=1,
            soft=True),

        # Overall standard gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="overall",
            selected_labels=None,
            alpha=1,
            beta=2,
            soft=False),

        # Overall soft gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="overall",
            selected_labels=None,
            alpha=1,
            beta=2,
            soft=True),

        # Arguemntation standard gamma
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=1,
            soft=False),

        # Arguemntation soft gamma
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=1,
            soft=True),

        # Arguemntation standard gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=2,
            soft=False),

        # Arguemntation soft gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=2,
            soft=True),

        # Narration standard gamma
        compute_gamma_test(
            all_annotations,
            layer_name="narration",
            selected_labels=narration_labels,
            alpha=1,
            beta=1,
            soft=False,
            add_none_for_missing_narration=True),


        # Narration soft gamma
        compute_gamma_test(
            all_annotations,
            layer_name="narration",
            selected_labels=narration_labels,
            alpha=1,
            beta=1,
            soft=True,
            add_none_for_missing_narration=True),

        # Narration standard gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="narration",
            selected_labels=narration_labels,
            alpha=1,
            beta=2,
            soft=False,
            add_none_for_missing_narration=True),

        # Narration soft gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="narration",
            selected_labels=narration_labels,
            alpha=1,
            beta=2,
            soft=True,
            add_none_for_missing_narration=True),
    ]


    for result in gamma_tests:
        result["document"] = document_id
        result["luiza_annotations"] = len(luiza_annotations)
        result["sara_annotations"] = len(sara_annotations)
        all_results.append(result)

Processing: TED_004
Processing: TED_014
Processing: TED_003
Processing: TED_002
Processing: TED_007
Processing: TED_012
Processing: TED_005
Processing: TED_017
Processing: TED_006
Processing: TED_019
Processing: TED_015
Processing: TED_001
Processing: TED_009
Processing: TED_010
Processing: TED_011
Processing: TED_018
Processing: TED_020
Processing: TED_008
Processing: TED_016
Processing: TED_013


# Save all result in a table

In [13]:
all_results_df = pd.DataFrame(all_results)

all_results_df = all_results_df[
    [
        "document",
        "layer",
        "gamma_type",
        "alpha",
        "beta",
        "gamma",
        "luiza_annotations",
        "sara_annotations"
    ]
]

all_results_df

,document,layer,gamma_type,alpha,beta,gamma,luiza_annotations,sara_annotations
0,TED_004,overall,standard,1,1,0.211150,45,18
1,TED_004,overall,soft,1,1,0.273660,45,18
2,TED_004,overall,standard,1,2,0.239975,45,18
3,TED_004,overall,soft,1,2,0.295891,45,18
4,TED_004,argumentation,standard,1,1,0.303924,45,18
...,...,...,...,...,...,...,...,...
235,TED_013,argumentation,soft,1,2,0.860491,43,31
236,TED_013,narration,standard,1,1,0.566615,43,31
237,TED_013,narration,soft,1,1,0.726110,43,31
238,TED_013,narration,standard,1,2,0.594423,43,31


In [14]:
output_path = "/content/drive/MyDrive/ALL_gamma_results_teds.csv"

all_results_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /content/drive/MyDrive/ALL_gamma_results_teds.csv


# EXTRA: only argumentation Gamma

In [15]:
only_arg_results = []

for pair in document_pairs:
    document_id = pair["document"]

    print("Processing:", document_id)

    luiza_annotations = parse_inception_json(
        pair["luiza_file"],
        "Luiza"
    )

    sara_annotations = parse_inception_json(
        pair["sara_file"],
        "Sara"
    )

    all_annotations = luiza_annotations + sara_annotations

    gamma_tests = [

        # Arguemntation standard gamma
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=1,
            soft=False),

        # Arguemntation soft gamma
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=1,
            soft=True),

        # Arguemntation standard gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=2,
            soft=False),

        # Arguemntation soft gamma label stronger
        compute_gamma_test(
            all_annotations,
            layer_name="argumentation",
            selected_labels=argumentation_labels,
            alpha=1,
            beta=2,
            soft=True),
    ]


    for result in gamma_tests:
        result["document"] = document_id
        result["luiza_annotations"] = len(luiza_annotations)
        result["sara_annotations"] = len(sara_annotations)
        only_arg_results.append(result)

Processing: TED_004
Processing: TED_014
Processing: TED_003
Processing: TED_002
Processing: TED_007
Processing: TED_012
Processing: TED_005
Processing: TED_017
Processing: TED_006
Processing: TED_019
Processing: TED_015
Processing: TED_001
Processing: TED_009
Processing: TED_010
Processing: TED_011
Processing: TED_018
Processing: TED_020
Processing: TED_008
Processing: TED_016
Processing: TED_013


# Save in a table

In [16]:
only_arg_results_df = pd.DataFrame(all_results)

only_arg_results_df = only_arg_results_df[
    [
        "document",
        "layer",
        "gamma_type",
        "alpha",
        "beta",
        "gamma",
        "luiza_annotations",
        "sara_annotations"
    ]
]

only_arg_results_df

,document,layer,gamma_type,alpha,beta,gamma,luiza_annotations,sara_annotations
0,TED_004,overall,standard,1,1,0.211150,45,18
1,TED_004,overall,soft,1,1,0.273660,45,18
2,TED_004,overall,standard,1,2,0.239975,45,18
3,TED_004,overall,soft,1,2,0.295891,45,18
4,TED_004,argumentation,standard,1,1,0.303924,45,18
...,...,...,...,...,...,...,...,...
235,TED_013,argumentation,soft,1,2,0.860491,43,31
236,TED_013,narration,standard,1,1,0.566615,43,31
237,TED_013,narration,soft,1,1,0.726110,43,31
238,TED_013,narration,standard,1,2,0.594423,43,31


In [17]:
output_path = "/content/drive/MyDrive/ONLY_ARG_gamma_results_teds.csv"

only_arg_results_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /content/drive/MyDrive/ONLY_ARG_gamma_results_teds.csv
